# 15 · Arrays, structs, maps and JSON (Spark)

Real data (events, APIs, BigQuery exports) is often **nested**:
- **array**: a list of values in one cell, e.g. `["Mug", "Cap"]`
- **struct**: a small record in one cell, e.g. `{city: "Austin", zip: "73301"}`, read with `col("address.city")`
- **map**: key → value pairs, e.g. `{"color": "red"}`

Key tools: `collect_list` (rows → array), `explode` (array → rows), `from_json` / `to_json`.
BigQuery has the same ideas: `ARRAY_AGG`, `UNNEST`, `STRUCT`.

**How to run:** Start a cell with `%%sql` to write Spark SQL. In a normal Python cell use the DataFrame API: the tables are ready as variables (`employees`, `orders`, ...). Put your answer in `res` and call `res.show()`.

Tables: see [tables_guide.md](../../jarvis/tables_guide.md). Run the setup cell first (Shift+Enter).

> **Solutions.** Try the practice notebook first: [`spark/practice/15_complex_types.ipynb`](../../spark/practice/15_complex_types.ipynb)

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "settings.py").exists())   # the repo folder
sys.path.insert(0, str(ROOT / "spark"))
from spark_setup import get_spark, load_tables
from pyspark.sql import functions as F, Window, types as T

spark = get_spark()
t = load_tables(spark)          # also makes SQL views with the same names
employees, departments = t["employees"], t["departments"]
customers, orders = t["customers"], t["orders"]
products, order_items = t["products"], t["order_items"]
user_logins = t["user_logins"]
%load_ext spark_setup

## Q1. Rows to an array (collect_list)  ·  🟢 easy

Make `baskets`: one row per `order_id` with `items` = the sorted list of product names in that order. Show orders 1001–1003.

*Expected result: 3 rows · columns: `order_id`, `items`*

**How it works:** groupBy order, `collect_list(product_name)` gathers names into an array, `sort_array` sorts it.

In [ ]:
%%sql
SELECT i.order_id, SORT_ARRAY(COLLECT_LIST(p.product_name)) AS items
FROM order_items i
JOIN products p ON p.product_id = i.product_id
WHERE i.order_id <= 1003
GROUP BY i.order_id
ORDER BY i.order_id

,order_id,items
0,1001,"[SQL Basics, Statistics Made Easy, Water Bottle]"
1,1002,[Data Engineering 101]
2,1003,"[Cap, Data Engineering 101, Socks Pack]"


**DataFrame API:**

In [ ]:
baskets = (order_items.join(products, "product_id")
           .groupBy("order_id")
           .agg(F.sort_array(F.collect_list("product_name")).alias("items")))
res = baskets.filter(F.col("order_id") <= 1003).orderBy("order_id")
res.show()

+--------+--------------------+
|order_id|               items|
+--------+--------------------+
|    1001|[SQL Basics, Stat...|
|    1002|[Data Engineering...|
|    1003|[Cap, Data Engine...|
+--------+--------------------+

**Why / traps:** `collect_list` keeps duplicates, `collect_set` removes them. The order inside the list is **not guaranteed**: sort it if it matters.

## Q2. Array to rows (explode)  ·  🟢 easy

From `baskets`, make one row per (order_id, item) again, with the **position** of each item. Show orders 1001–1002.

*Expected result: 4 rows · columns: `order_id`, `pos`, `item`*

**How it works:** `posexplode` turns each array element into its own row, with its position.

In [ ]:
%%sql
WITH baskets AS (
    SELECT i.order_id, SORT_ARRAY(COLLECT_LIST(p.product_name)) AS items
    FROM order_items i JOIN products p ON p.product_id = i.product_id
    GROUP BY i.order_id
)
SELECT order_id, pos, item
FROM baskets
LATERAL VIEW POSEXPLODE(items) x AS pos, item
WHERE order_id <= 1002
ORDER BY order_id, pos

,order_id,pos,item
0,1001,0,SQL Basics
1,1001,1,Statistics Made Easy
2,1001,2,Water Bottle
3,1002,0,Data Engineering 101


**DataFrame API:**

In [ ]:
baskets = (order_items.join(products, "product_id")
           .groupBy("order_id")
           .agg(F.sort_array(F.collect_list("product_name")).alias("items")))
res = (baskets.filter("order_id <= 1002")
       .select("order_id", F.posexplode("items").alias("pos", "item"))
       .orderBy("order_id", "pos"))
res.show()

+--------+---+--------------------+
|order_id|pos|                item|
+--------+---+--------------------+
|    1001|  0|          SQL Basics|
|    1001|  1|Statistics Made Easy|
|    1001|  2|        Water Bottle|
|    1002|  0|Data Engineering 101|
+--------+---+--------------------+

**Why / traps:** `explode` drops rows whose array is empty or NULL. `explode_outer` keeps them (with NULL).

## Q3. Array functions  ·  🟡 medium

For each basket show `order_id`, `n_items` (size), `has_mug` (contains `Coffee Mug`), and `upper_items` (every name in capitals, with `transform`). Only baskets with 3+ items.

**How it works:** Array functions work on the whole list: `size`, `array_contains`, and `transform` applies a function to each element.

In [ ]:
baskets = (order_items.join(products, "product_id")
           .groupBy("order_id")
           .agg(F.sort_array(F.collect_list("product_name")).alias("items")))
res = (baskets
       .select("order_id",
               F.size("items").alias("n_items"),
               F.array_contains("items", "Coffee Mug").alias("has_mug"),
               F.transform("items", lambda x: F.upper(x)).alias("upper_items"))
       .filter("n_items >= 3").orderBy("order_id"))
res.show()

+--------+-------+-------+--------------------+
|order_id|n_items|has_mug|         upper_items|
+--------+-------+-------+--------------------+
|    1001|      3|  false|[SQL BASICS, STAT...|
|    1003|      3|  false|[CAP, DATA ENGINE...|
|    1006|      3|  false|[DATA ENGINEERING...|
|    1012|      3|  false|[LAPTOP STAND, US...|
|    1015|      3|  false|[HOODIE, SQL BASI...|
|    1016|      3|   true|[COFFEE MUG, SOCK...|
|    1017|      3|  false|[BLUETOOTH SPEAKE...|
|    1019|      3|  false|[BLUETOOTH SPEAKE...|
|    1022|      3|  false|[BLUETOOTH SPEAKE...|
|    1025|      3|  false|[LAPTOP STAND, ST...|
|    1045|      3|  false|[PYTHON COOKBOOK,...|
|    1046|      3|  false|[DESK LAMP, NOTEB...|
|    1050|      3|  false|[CAP, STATISTICS ...|
|    1053|      3|   true|[BLUETOOTH SPEAKE...|
|    1056|      3|  false|[HOODIE, SQL BASI...|
|    1058|      3|  false|[DATA ENGINEERING...|
|    1059|      3|   true|[COFFEE MUG, SOCK...|
|    1060|      3|  false|[HEADPHONES, H

**Why / traps:** **Higher-order functions** (`transform`, `filter`, `exists`, `aggregate`) work on each element **inside** the array, with no explode and no UDF. Fast.

## Q4. Build and read a struct  ·  🟢 easy

Make `customers_nested` with columns `customer_id` and `info` = struct(name, city, signup_date). Print the schema, then select `info.city` back out for customers 1–3.

*Expected result: 3 rows · columns: `customer_id`, `city`*

**How it works:** `F.struct(...)` packs columns into one struct column; `col("info.city")` reads one field back.

In [ ]:
customers_nested = customers.select("customer_id", F.struct("name", "city", "signup_date").alias("info"))
customers_nested.printSchema()
res = customers_nested.filter("customer_id <= 3").select("customer_id", F.col("info.city").alias("city"))
res.show()

root
 |-- customer_id: integer (nullable = true)
 |-- info: struct (nullable = false)
 |    |-- name: string (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- signup_date: date (nullable = true)

+-----------+-------+
|customer_id|   city|
+-----------+-------+
|          1|Chicago|
|          2|Seattle|
|          3|Phoenix|
+-----------+-------+

**Why / traps:** `info.*` expands all fields of a struct into columns: `select("customer_id", "info.*")`.

## Q5. Maps  ·  🟡 medium

For products 1–3 build `attrs` = map(`category` → category, `price` → price as string). Then show `product_id`, the map, and `attrs['category']`.

*Expected result: 3 rows · columns: `product_id`, `attrs`, `cat`*

**How it works:** `create_map(key1, value1, key2, value2)` builds a map; `col("attrs")["category"]` reads one key.

In [ ]:
res = (products.filter("product_id <= 3")
       .select("product_id",
               F.create_map(F.lit("category"), F.col("category"),
                            F.lit("price"), F.col("price").cast("string")).alias("attrs"))
       .withColumn("cat", F.col("attrs")["category"]))
res.show()

+----------+--------------------+-----------+
|product_id|               attrs|        cat|
+----------+--------------------+-----------+
|         1|{category -> Elec...|Electronics|
|         2|{category -> Elec...|Electronics|
|         3|{category -> Elec...|Electronics|
+----------+--------------------+-----------+

**Why / traps:** All map values must have **one type**, so the price is cast to string. Use a struct when fields have different types.

## Q6. Parse JSON text (from_json)  ·  🟡 medium

A raw table has JSON text in a column. Parse it with a schema and pull out the fields: `event`, `user.id`, and the number of `tags`.

*Expected result: 3 rows · columns: `event`, `user_id`, `n_tags`*

**How it works:** `from_json(text, schema)` turns JSON text into a struct; then read fields with dots. Bad JSON → NULL.

In [ ]:
raw = spark.createDataFrame([
    ('{"event": "login",  "user": {"id": 1, "country": "IN"}, "tags": ["web", "new"]}',),
    ('{"event": "logout", "user": {"id": 2, "country": "UK"}, "tags": []}',),
    ('not json at all',),
], "value string")

schema = "event STRING, user STRUCT<id: INT, country: STRING>, tags ARRAY<STRING>"
parsed = raw.withColumn("j", F.from_json("value", schema))
res = parsed.select(F.col("j.event").alias("event"),
                    F.col("j.user.id").alias("user_id"),
                    F.size("j.tags").alias("n_tags"))
res.show()

+------+-------+------+
| event|user_id|n_tags|
+------+-------+------+
| login|      1|     2|
|logout|      2|     0|
|  NULL|   NULL|  NULL|
+------+-------+------+

**Why / traps:** Bad JSON gives NULL fields (no error). `get_json_object(value, '$.user.id')` reads one path without a schema. `schema_of_json` can suggest a schema from a sample.

## Q7. Back to JSON (to_json)  ·  🟡 medium

For departments 1–2 make one JSON text per department: `{"dept": ..., "people": [names...]}`.

**How it works:** Build a struct (dept + list of people), then `to_json` turns it into a JSON string.

In [ ]:
res = (employees.filter("dept_id <= 2").join(departments, "dept_id")
       .groupBy("dept_name").agg(F.sort_array(F.collect_list("name")).alias("people"))
       .select(F.to_json(F.struct(F.col("dept_name").alias("dept"), "people")).alias("json")))
for row in res.collect():
    print(row.json)
res.show()

{"dept":"Sales","people":["Henry Clark","Irene Lewis","Jack Walker","Karen Hall","Leo Allen","Mia Young"]}
{"dept":"Engineering","people":["Alice Johnson","Bob Smith","Carol White","David Brown","Eva Green","Frank Moore","Grace Lee"]}

+--------------------+
|                json|
+--------------------+
|{"dept":"Sales","...|
|{"dept":"Engineer...|
+--------------------+

**Why / traps:** Common when you send data to an API or a message queue (e.g. Pub/Sub).

## Q8. Flatten nested data  ·  🔴 hard

Turn this nested order into flat rows (one per line item): order_id, customer.name, item.sku, item.qty.

*Expected result: 3 rows · columns: `order_id`, `customer_name`, `sku`, `qty`*

**How it works:** `explode` the lines array into rows, then select fields of the struct with dots.

In [ ]:
%%sql
WITH nested AS (
    SELECT 1 AS order_id, NAMED_STRUCT('name', 'Aarav', 'city', 'Chicago') AS customer,
           ARRAY(NAMED_STRUCT('sku', 'SKU-1', 'qty', 2), NAMED_STRUCT('sku', 'SKU-7', 'qty', 1)) AS lines
    UNION ALL
    SELECT 2, NAMED_STRUCT('name', 'Priya', 'city', 'Seattle'),
           ARRAY(NAMED_STRUCT('sku', 'SKU-3', 'qty', 5))
)
SELECT order_id, customer.name AS customer_name, line.sku, line.qty
FROM nested
LATERAL VIEW EXPLODE(lines) t AS line

,order_id,customer_name,sku,qty
0,1,Aarav,SKU-1,2
1,1,Aarav,SKU-7,1
2,2,Priya,SKU-3,5


**DataFrame API:**

In [ ]:
nested = spark.createDataFrame(
    [(1, ("Aarav", "Chicago"), [("SKU-1", 2), ("SKU-7", 1)]),
     (2, ("Priya", "Seattle"), [("SKU-3", 5)])],
    "order_id int, customer struct<name:string, city:string>, lines array<struct<sku:string, qty:int>>")
res = (nested
       .select("order_id", F.col("customer.name").alias("customer_name"), F.explode("lines").alias("line"))
       .select("order_id", "customer_name", "line.sku", "line.qty"))
res.show()

+--------+-------------+-----+---+
|order_id|customer_name|  sku|qty|
+--------+-------------+-----+---+
|       1|        Aarav|SKU-1|  2|
|       1|        Aarav|SKU-7|  1|
|       2|        Priya|SKU-3|  5|
+--------+-------------+-----+---+

**Why / traps:** Pattern: **explode** the arrays, then **select struct fields** with dots. In BigQuery: `FROM t, UNNEST(t.lines) AS line`.

## Q9. Zip two arrays  ·  🟡 medium

Each row has a list of `skus` and a list of `qtys` of the same length. Pair them up and explode to rows (sku, qty).

*Expected result: 3 rows · columns: `order_id`, `sku`, `qty`*

**How it works:** `arrays_zip` pairs elements by position into structs; `explode` makes one row per pair.

In [ ]:
%%sql
SELECT order_id, z.skus AS sku, z.qtys AS qty
FROM VALUES (1, ARRAY('A', 'B'), ARRAY(2, 5)), (2, ARRAY('C'), ARRAY(1)) AS t(order_id, skus, qtys)
LATERAL VIEW EXPLODE(ARRAYS_ZIP(skus, qtys)) x AS z

,order_id,sku,qty
0,1,A,2
1,1,B,5
2,2,C,1


**DataFrame API:**

In [ ]:
df = spark.createDataFrame([(1, ["A", "B"], [2, 5]), (2, ["C"], [1])],
                           "order_id int, skus array<string>, qtys array<int>")
res = (df.select("order_id", F.explode(F.arrays_zip("skus", "qtys")).alias("z"))
         .select("order_id", F.col("z.skus").alias("sku"), F.col("z.qtys").alias("qty")))
res.show()

+--------+---+---+
|order_id|sku|qty|
+--------+---+---+
|       1|  A|  2|
|       1|  B|  5|
|       2|  C|  1|
+--------+---+---+